# NB_ingest_to_hubs — Silver → Hubs

Reads each Silver source table defined in the DV model config,
computes the SHA-256 hash key from business key columns, and performs
an INSERT-only MERGE into the corresponding vault hub table.
No updates — hubs are insert-only by DV 2.0 design.

In [ ]:
import json
from pathlib import Path

def load_model(model_path: str) -> dict:
    """Load dv_model.json and return as dict."""
    p = Path(model_path)
    if not p.is_absolute():
        # Try relative to the repo root (Databricks workspace)
        p = Path('/Workspace/Repos') / p
    if p.exists():
        return json.loads(p.read_text())
    # Fallback: dbutils.fs.head for DBFS paths
    content = dbutils.fs.head(f'dbfs:/{model_path}', 1_000_000)
    return json.loads(content)

from pyspark.sql import functions as F

def generate_hash_key(bk_cols: list) -> F.Column:
    """Generate SHA-256 hash key from business key columns."""
    return F.sha2(
        F.concat_ws("||", *[F.upper(F.trim(F.col(c).cast("string"))) for c in bk_cols]),
        256,
    )

def generate_diff_hash(tracked_cols: list) -> F.Column:
    """Generate SHA-256 diff hash from tracked satellite columns."""
    return F.sha2(
        F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NULL")) for c in tracked_cols]),
        256,
    )

def create_hub_table(hub_cfg: dict) -> None:
    """Create a hub Delta table if it does not exist."""
    table = f"{CATALOG}.{VAULT_SCHEMA}.{hub_cfg['name'].lower()}"
    bk_cols = ', '.join(f"{c} STRING" for c in hub_cfg['business_key_columns'])
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            HK_{hub_cfg['name'][4:]} STRING NOT NULL,
            {bk_cols},
            LOAD_DATE TIMESTAMP,
            RECORD_SOURCE STRING
        ) USING DELTA
    """)

def create_link_table(lnk_cfg: dict) -> None:
    """Create a link Delta table if it does not exist."""
    table = f"{CATALOG}.{VAULT_SCHEMA}.{lnk_cfg['name'].lower()}"
    fk_cols = ', '.join(f"HK_{r['hub'][4:]} STRING" for r in lnk_cfg['hub_references'])
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            HK_{lnk_cfg['name'][4:]} STRING NOT NULL,
            {fk_cols},
            LOAD_DATE TIMESTAMP,
            RECORD_SOURCE STRING
        ) USING DELTA
    """)

def create_sat_table(sat_cfg: dict) -> None:
    """Create a satellite Delta table if it does not exist."""
    table = f"{CATALOG}.{VAULT_SCHEMA}.{sat_cfg['name'].lower()}"
    tracked_cols = ', '.join(f"{c} STRING" for c in sat_cfg['tracked_columns'])
    parent_hub = sat_cfg['parent_hub']
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            HK_{parent_hub[4:]} STRING NOT NULL,
            LOAD_DATE TIMESTAMP NOT NULL,
            DIFF_HK STRING,
            {tracked_cols},
            RECORD_SOURCE STRING
        ) USING DELTA
    """)

def create_pit_table(pit_cfg: dict) -> None:
    """Create a PIT Delta table if it does not exist."""
    table = f"{CATALOG}.{VAULT_SCHEMA}.{pit_cfg['name'].lower()}"
    hub_name = pit_cfg['hub']
    sat_cols = '\n'.join(f"    {s}_LDTS TIMESTAMP,\n    {s}_HK STRING," for s in pit_cfg['satellites'])
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            HK_{hub_name[4:]} STRING NOT NULL,
            SNAPSHOT_DATE DATE NOT NULL,
            {sat_cols}
            LOAD_DATE TIMESTAMP
        ) USING DELTA
    """)

def create_bridge_table(brg_cfg: dict) -> None:
    """Create a bridge Delta table if it does not exist."""
    table = f"{CATALOG}.{VAULT_SCHEMA}.{brg_cfg['name'].lower()}"
    hk_cols = '\n'.join(f"    HK_{p[4:]} STRING," for p in brg_cfg['path'] if p.startswith('HUB_'))
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            {hk_cols}
            LOAD_DATE TIMESTAMP
        ) USING DELTA
    """)

from pyspark.sql import Window

def get_latest_diff_hash(sat_table: str, hk_col: str) -> 'DataFrame':
    """Return the latest DIFF_HK per hub key from an existing satellite table.

    Uses a window function to find the most recent record per hub key,
    which is then LEFT JOINed against incoming data to detect changes.
    """
    w = Window.partitionBy(hk_col).orderBy(F.col('LOAD_DATE').desc())
    return (
        spark.table(sat_table)
        .withColumn('_rn', F.row_number().over(w))
        .filter(F.col('_rn') == 1)
        .select(hk_col, 'DIFF_HK')
    )

In [ ]:
dbutils.widgets.text("CATALOG", "workspace", "Unity Catalog name")
dbutils.widgets.text("VAULT_SCHEMA", "vault", "Vault schema")
dbutils.widgets.text("SILVER_SCHEMA", "silver", "Silver schema")
dbutils.widgets.text("MODEL_PATH", "pipeline_configs/datavault/dv_model.json", "DV Model JSON Path")
dbutils.widgets.text("WATERMARK_TS", "", "Optional: only process records >= this timestamp")
dbutils.widgets.text("FULL_RELOAD", "false", "Drop and recreate hub tables before loading")

CATALOG       = dbutils.widgets.get("CATALOG")
VAULT_SCHEMA  = dbutils.widgets.get("VAULT_SCHEMA")
SILVER_SCHEMA = dbutils.widgets.get("SILVER_SCHEMA")
MODEL_PATH    = dbutils.widgets.get("MODEL_PATH")
WATERMARK_TS  = dbutils.widgets.get("WATERMARK_TS")
FULL_RELOAD   = dbutils.widgets.get("FULL_RELOAD").strip().lower() == "true"

model = load_model(MODEL_PATH)
print(f"Loaded {len(model['hubs'])} hubs from model")
if FULL_RELOAD:
    print("FULL_RELOAD=true: hub tables will be dropped and recreated")

In [ ]:
model = load_model(MODEL_PATH)
print(f"Loaded {len(model['hubs'])} hubs from model")

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

hub_counts = {}

for hub_cfg in model['hubs']:
    if not hub_cfg.get('enabled', True):
        print(f"  Skipping disabled hub: {hub_cfg['name']}")
        continue

    hub_name    = hub_cfg['name']
    source_tbl  = hub_cfg['source_table']
    bk_cols     = hub_cfg['business_key_columns']
    load_dt_col = hub_cfg['load_date_column']
    rec_src     = hub_cfg['record_source']
    hk_alias    = f"HK_{hub_name[4:]}"
    target_tbl  = f"{CATALOG}.{VAULT_SCHEMA}.{hub_name.lower()}"

    if FULL_RELOAD:
        spark.sql(f"DROP TABLE IF EXISTS {target_tbl}")
        print(f"  Dropped {target_tbl}")

    # Ensure target table exists
    create_hub_table(hub_cfg)

    # Read Silver source
    src_df = spark.table(source_tbl)
    if WATERMARK_TS:
        wm_col = 'last_updated_dt' if 'last_updated_dt' in src_df.columns else bk_cols[0]
        src_df = src_df.filter(F.col(wm_col) >= WATERMARK_TS)

    # Compute hash key
    src_df = (
        src_df
        .withColumn(hk_alias, generate_hash_key(bk_cols))
        .withColumn('LOAD_DATE', F.current_timestamp())
        .withColumn('RECORD_SOURCE', F.lit(rec_src))
        .select(hk_alias, *bk_cols, 'LOAD_DATE', 'RECORD_SOURCE')
        .dropDuplicates([hk_alias])
    )

    # INSERT-only MERGE (DV 2.0: hubs never update)
    hub_tbl = DeltaTable.forName(spark, target_tbl)
    (
        hub_tbl.alias('tgt')
        .merge(src_df.alias('src'), f"tgt.{hk_alias} = src.{hk_alias}")
        .whenNotMatchedInsertAll()
        .execute()
    )

    count = spark.table(target_tbl).count()
    hub_counts[hub_name] = count
    print(f"  {hub_name}: {count:,} total rows in {target_tbl}")

In [ ]:
print('\nHub ingestion complete.')
for hub_name, cnt in hub_counts.items():
    print(f'  {hub_name}: {cnt:,} rows')